# Chapter 6.8: Custom CUDA Kernels in SGLang

This notebook examines the custom CUDA and Triton kernels used in SGLang to accelerate LLM inference. We cover attention kernels, fused activations, quantization kernels, communication primitives, and kernel dispatch logic.

## Learning Objectives
1. Survey custom ops in the SGLang codebase
2. Understand FlashInfer integration for attention
3. Implement fused activation kernels in Triton
4. Explore quantization kernels (FP8/INT8)
5. Understand kernel dispatch and selection

## Prerequisites
- CUDA programming concepts (threads, blocks, warps)
- PyTorch custom operators basics
- Linear algebra for transformers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part6/chapter_6.8_cuda_kernels.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part6/chapter_6.8_cuda_kernels.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Tuple
import time
import math

# Check if Triton is available
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
    print(f"Triton version: {triton.__version__}")
except ImportError:
    HAS_TRITON = False
    print("Triton not available. Will use PyTorch equivalents for demonstrations.")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

---
## Part 1: Overview of Custom Ops in SGLang

SGLang uses custom kernels for performance-critical operations. Here's a map:

```
SGLang Custom Kernels
├── Attention Kernels
│   ├── FlashInfer (primary backend)
│   ├── FlashAttention-2 (alternative)
│   └── Paged attention (vLLM-style)
├── Fused Activation Kernels
│   ├── SiLU × mul (Llama-style FFN)
│   ├── GeGLU (Gemma-style)
│   └── Fused add + RMSNorm
├── Quantization Kernels
│   ├── FP8 dynamic quantization
│   ├── INT8 W8A8 GEMM
│   ├── GPTQ/AWQ dequantization
│   └── Marlin kernel (fast 4-bit)
├── Communication Kernels
│   ├── Custom all-reduce (for TP)
│   ├── All-gather
│   └── NCCL wrappers
└── Sampling Kernels
    ├── Top-k/top-p (fused)
    ├── Multinomial sampling
    └── Penalties (fused)
```

### Why Custom Kernels?

PyTorch's built-in ops launch a separate CUDA kernel for each operation. For a sequence like `y = RMSNorm(x + residual)`, PyTorch would:
1. Launch kernel for `x + residual` → read+write to GPU memory
2. Launch kernel for variance computation → read+write
3. Launch kernel for normalization → read+write

A **fused kernel** does all three in one pass, reading input once and writing output once. This reduces memory bandwidth by ~3x.

In [ ]:
# Demonstrate why kernel fusion matters

def rmsnorm_unfused(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Unfused RMSNorm: 3 separate operations."""
    # Op 1: compute variance
    variance = x.pow(2).mean(-1, keepdim=True)
    # Op 2: normalize
    x_norm = x * torch.rsqrt(variance + eps)
    # Op 3: scale
    return weight * x_norm


def rmsnorm_with_residual_unfused(
    x: torch.Tensor, residual: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Unfused: residual add + RMSNorm (5 ops)."""
    # Op 1: residual addition
    hidden = x + residual
    # Op 2-4: RMSNorm
    normed = rmsnorm_unfused(hidden, weight, eps)
    return normed, hidden


def rmsnorm_with_residual_fused(
    x: torch.Tensor, residual: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Simulated fused kernel: add + RMSNorm in one pass.
    
    In a real fused CUDA kernel, this would:
    1. Read x and residual once from global memory
    2. Compute x + residual in registers
    3. Compute variance using shared memory reduction
    4. Normalize and scale in registers
    5. Write output and updated residual once
    
    Memory reads: 2 (x, residual) instead of 5+
    Memory writes: 2 (output, hidden) instead of 4+
    """
    # In PyTorch this is still unfused, but represents the fused logic
    hidden = x + residual
    variance = hidden.pow(2).mean(-1, keepdim=True)
    normed = weight * hidden * torch.rsqrt(variance + eps)
    return normed, hidden


# Benchmark
hidden_size = 4096
seq_len = 2048
x = torch.randn(seq_len, hidden_size, device=DEVICE)
residual = torch.randn(seq_len, hidden_size, device=DEVICE)
weight = torch.ones(hidden_size, device=DEVICE)

# Warmup
for _ in range(10):
    _ = rmsnorm_with_residual_unfused(x, residual, weight)
    _ = rmsnorm_with_residual_fused(x, residual, weight)

# Time
n_iters = 100
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()

start = time.time()
for _ in range(n_iters):
    _ = rmsnorm_with_residual_unfused(x, residual, weight)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()
unfused_time = (time.time() - start) / n_iters * 1000

start = time.time()
for _ in range(n_iters):
    _ = rmsnorm_with_residual_fused(x, residual, weight)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()
fused_time = (time.time() - start) / n_iters * 1000

# Verify correctness
out_unfused, hid_unfused = rmsnorm_with_residual_unfused(x, residual, weight)
out_fused, hid_fused = rmsnorm_with_residual_fused(x, residual, weight)
error = (out_unfused - out_fused).abs().max().item()

print(f"=== RMSNorm + Residual Benchmark ===")
print(f"Shape: ({seq_len}, {hidden_size}), Device: {DEVICE}")
print(f"Unfused: {unfused_time:.3f} ms")
print(f"Fused:   {fused_time:.3f} ms")
print(f"Speedup: {unfused_time/fused_time:.2f}x")
print(f"Max error: {error:.2e}")
print(f"\nNote: On CPU the difference is small. On GPU, fused kernels are 2-3x faster")
print(f"due to reduced memory bandwidth usage.")

---
## Part 2: FlashInfer — SGLang's Attention Backend

SGLang primarily uses **FlashInfer** for attention computation. FlashInfer provides:
- Paged KV-cache attention (like PagedAttention but faster)
- Ragged tensor operations for variable-length sequences
- Cascading attention for prefix sharing
- FP8 KV-cache support

In [ ]:
# Demonstrate attention computation patterns

def standard_attention(
    q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
    mask: Optional[torch.Tensor] = None
) -> torch.Tensor:
    """Standard scaled dot-product attention.
    
    This is the naive O(N^2) implementation.
    q, k, v shape: (batch, heads, seq_len, head_dim)
    """
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, v)


def flash_attention_simulation(
    q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
    block_size: int = 64
) -> torch.Tensor:
    """Simulated FlashAttention: block-wise computation.
    
    FlashAttention processes attention in blocks to stay within SRAM.
    Key insight: softmax can be computed incrementally (online softmax trick).
    
    Real FlashAttention uses tiling in CUDA. This demonstrates the algorithm.
    """
    batch, heads, seq_len, head_dim = q.shape
    scale = 1.0 / math.sqrt(head_dim)
    
    # Output accumulator
    output = torch.zeros_like(q)
    # Running max and sum for online softmax
    row_max = torch.full((batch, heads, seq_len, 1), float('-inf'), device=q.device)
    row_sum = torch.zeros(batch, heads, seq_len, 1, device=q.device)
    
    # Process K,V in blocks
    for j_start in range(0, seq_len, block_size):
        j_end = min(j_start + block_size, seq_len)
        k_block = k[:, :, j_start:j_end, :]  # (B, H, block, D)
        v_block = v[:, :, j_start:j_end, :]  # (B, H, block, D)
        
        # Compute attention scores for this block
        scores = torch.matmul(q, k_block.transpose(-2, -1)) * scale  # (B, H, N, block)
        
        # Online softmax: update running max and sum
        block_max = scores.max(dim=-1, keepdim=True).values
        new_max = torch.maximum(row_max, block_max)
        
        # Rescale previous accumulator
        exp_old = torch.exp(row_max - new_max)
        exp_new = torch.exp(scores - new_max)
        
        new_sum = row_sum * exp_old + exp_new.sum(dim=-1, keepdim=True)
        
        # Update output
        output = output * (row_sum * exp_old / new_sum) + \
                 torch.matmul(exp_new, v_block) / new_sum
        
        row_max = new_max
        row_sum = new_sum
    
    return output


# Verify correctness
batch, heads, seq_len, head_dim = 2, 8, 128, 64
q = torch.randn(batch, heads, seq_len, head_dim)
k = torch.randn(batch, heads, seq_len, head_dim)
v = torch.randn(batch, heads, seq_len, head_dim)

out_standard = standard_attention(q, k, v)
out_flash = flash_attention_simulation(q, k, v, block_size=32)

error = (out_standard - out_flash).abs().max().item()
print(f"=== Attention Verification ===")
print(f"Shape: batch={batch}, heads={heads}, seq_len={seq_len}, head_dim={head_dim}")
print(f"Max error (standard vs flash simulation): {error:.6e}")
print(f"Correct: {'Yes' if error < 1e-4 else 'No'}")

In [ ]:
# Memory analysis: standard vs flash attention

def attention_memory_analysis(seq_lengths: List[int], head_dim: int = 128, num_heads: int = 32):
    """Compare memory requirements of standard vs flash attention."""
    results = []
    
    for seq_len in seq_lengths:
        # Standard attention: needs full N×N attention matrix
        attn_matrix_bytes = seq_len * seq_len * num_heads * 4  # float32
        qkv_bytes = 3 * seq_len * head_dim * num_heads * 4
        standard_total = attn_matrix_bytes + qkv_bytes
        
        # Flash attention: only needs block_size × block_size per block
        block_size = 128  # Typical block size
        flash_working = block_size * block_size * num_heads * 4  # Shared memory
        flash_total = qkv_bytes + flash_working
        
        results.append({
            'seq_len': seq_len,
            'standard_mb': standard_total / (1024**2),
            'flash_mb': flash_total / (1024**2),
            'ratio': standard_total / flash_total
        })
    
    return results


seq_lengths = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768]
results = attention_memory_analysis(seq_lengths)

print(f"{'Seq Length':<12} {'Standard (MB)':<15} {'Flash (MB)':<12} {'Savings':<10}")
print("-" * 49)
for r in results:
    print(f"{r['seq_len']:<12} {r['standard_mb']:<15.1f} {r['flash_mb']:<12.1f} {r['ratio']:<10.1f}x")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(seq_lengths, [r['standard_mb'] for r in results], 'r-o', label='Standard Attention', linewidth=2)
ax.semilogy(seq_lengths, [r['flash_mb'] for r in results], 'b-o', label='Flash Attention', linewidth=2)
ax.set_xlabel('Sequence Length')
ax.set_ylabel('Memory (MB, log scale)')
ax.set_title('Attention Memory: O(N²) vs O(N)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 3: Fused Activation Kernels

Transformers use activation functions in their FFN layers. Fusing the activation with the element-wise multiply saves memory bandwidth.

### SiLU × mul (Llama, Mistral)
```python
# Unfused:
gate = silu(gate_proj(x))   # Write to memory
up = up_proj(x)             # Write to memory
output = gate * up          # Read both, write result

# Fused:
output = silu_and_mul(gate_proj(x), up_proj(x))  # One kernel, one write
```

In [ ]:
def silu(x: torch.Tensor) -> torch.Tensor:
    """SiLU (Swish) activation: x * sigmoid(x)"""
    return x * torch.sigmoid(x)


def silu_and_mul_unfused(gate: torch.Tensor, up: torch.Tensor) -> torch.Tensor:
    """Unfused SiLU×mul: 3 memory operations."""
    activated = silu(gate)    # Op 1: read gate, write activated
    return activated * up     # Op 2: read activated + up, write output


def silu_and_mul_fused(gate: torch.Tensor, up: torch.Tensor) -> torch.Tensor:
    """Simulated fused SiLU×mul: single operation.
    
    In a real CUDA kernel, this would:
    1. Read gate[i] and up[i] from global memory
    2. Compute silu(gate[i]) * up[i] in registers
    3. Write output[i] to global memory
    
    Saves one read + one write of intermediate tensor.
    """
    return (gate * torch.sigmoid(gate)) * up


def geglu_unfused(gate: torch.Tensor, up: torch.Tensor) -> torch.Tensor:
    """Unfused GeGLU activation (used by Gemma)."""
    return F.gelu(gate) * up


def geglu_fused(gate: torch.Tensor, up: torch.Tensor) -> torch.Tensor:
    """Simulated fused GeGLU."""
    return F.gelu(gate) * up


# Benchmark
sizes = [(1024, 4096), (2048, 11008), (4096, 14336)]  # (seq, ffn_dim)

print(f"{'Shape':<20} {'SiLU×mul unfused':<18} {'SiLU×mul fused':<18} {'Speedup':<10}")
print("-" * 66)

for seq, ffn in sizes:
    gate = torch.randn(seq, ffn, device=DEVICE)
    up = torch.randn(seq, ffn, device=DEVICE)
    
    # Warmup
    for _ in range(10):
        _ = silu_and_mul_unfused(gate, up)
        _ = silu_and_mul_fused(gate, up)
    
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    
    n_iters = 200
    start = time.time()
    for _ in range(n_iters):
        _ = silu_and_mul_unfused(gate, up)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    unfused_ms = (time.time() - start) / n_iters * 1000
    
    start = time.time()
    for _ in range(n_iters):
        _ = silu_and_mul_fused(gate, up)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    fused_ms = (time.time() - start) / n_iters * 1000
    
    print(f"({seq}, {ffn}){'':>8} {unfused_ms:<18.3f} {fused_ms:<18.3f} {unfused_ms/fused_ms:<10.2f}x")

In [ ]:
# Triton kernel implementation (if available)

if HAS_TRITON:
    @triton.jit
    def silu_and_mul_kernel(
        gate_ptr, up_ptr, output_ptr,
        n_elements,
        BLOCK_SIZE: tl.constexpr,
    ):
        """Triton kernel for fused SiLU × mul.
        
        Each program processes BLOCK_SIZE elements.
        """
        pid = tl.program_id(0)
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements
        
        # Load inputs
        gate = tl.load(gate_ptr + offsets, mask=mask)
        up = tl.load(up_ptr + offsets, mask=mask)
        
        # Compute SiLU(gate) * up
        sigmoid_gate = tl.sigmoid(gate)
        silu_gate = gate * sigmoid_gate
        output = silu_gate * up
        
        # Store output
        tl.store(output_ptr + offsets, output, mask=mask)
    
    def silu_and_mul_triton(gate: torch.Tensor, up: torch.Tensor) -> torch.Tensor:
        """Launch the Triton kernel."""
        assert gate.shape == up.shape
        output = torch.empty_like(gate)
        n_elements = gate.numel()
        grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
        silu_and_mul_kernel[grid](
            gate, up, output,
            n_elements,
            BLOCK_SIZE=1024,
        )
        return output
    
    # Test Triton kernel
    gate = torch.randn(2048, 4096, device='cuda')
    up = torch.randn(2048, 4096, device='cuda')
    
    out_pytorch = silu_and_mul_fused(gate, up)
    out_triton = silu_and_mul_triton(gate, up)
    
    error = (out_pytorch - out_triton).abs().max().item()
    print(f"Triton kernel error vs PyTorch: {error:.2e}")
    print("Triton kernel compiled and verified!")
else:
    print("Triton not available. Showing the kernel code:")
    triton_code = '''
@triton.jit
def silu_and_mul_kernel(
    gate_ptr, up_ptr, output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    
    gate = tl.load(gate_ptr + offsets, mask=mask)
    up = tl.load(up_ptr + offsets, mask=mask)
    
    sigmoid_gate = tl.sigmoid(gate)
    silu_gate = gate * sigmoid_gate
    output = silu_gate * up
    
    tl.store(output_ptr + offsets, output, mask=mask)
'''
    print(triton_code)

---
## Part 4: Quantization Kernels — FP8 and INT8

Quantization reduces memory and compute by using lower-precision data types. Custom kernels handle the quantize/dequantize + GEMM fusion.

In [ ]:
class FP8Quantizer:
    """Simulate FP8 dynamic quantization.
    
    FP8 E4M3 format: 1 sign + 4 exponent + 3 mantissa bits
    Range: [-448, 448], min positive: 2^-9
    
    In SGLang, FP8 quantization is used for:
    - KV cache (reduces memory by 2x vs FP16)
    - Weight quantization (with static scales)
    - Activation quantization (with dynamic scales)
    """
    FP8_E4M3_MAX = 448.0
    FP8_E4M3_MIN = 2**-9
    
    @staticmethod
    def dynamic_quantize(tensor: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Quantize a tensor to FP8 with dynamic scaling.
        
        Returns: (quantized_tensor, scale)
        In practice, the quantized tensor would be stored as FP8.
        We simulate using FP32 clamped to FP8 range.
        """
        # Compute per-tensor scale
        abs_max = tensor.abs().max()
        scale = abs_max / FP8Quantizer.FP8_E4M3_MAX
        scale = torch.clamp(scale, min=1e-12)
        
        # Quantize: divide by scale, clamp to FP8 range
        quantized = tensor / scale
        quantized = torch.clamp(quantized, -FP8Quantizer.FP8_E4M3_MAX, FP8Quantizer.FP8_E4M3_MAX)
        
        # Simulate FP8 precision loss (round to 3-bit mantissa)
        # FP8 E4M3 has 3 mantissa bits = 8 representable values per exponent
        quantized = torch.round(quantized * 8) / 8  # Crude simulation
        
        return quantized, scale
    
    @staticmethod
    def dequantize(quantized: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
        """Dequantize back to FP32/FP16."""
        return quantized * scale


class INT8Quantizer:
    """Simulate INT8 per-channel quantization.
    
    Used for W8A8 (8-bit weights, 8-bit activations) GEMM.
    """
    @staticmethod
    def quantize_per_channel(tensor: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Per-channel symmetric quantization to INT8."""
        abs_max = tensor.abs().amax(dim=-1, keepdim=True)
        scale = abs_max / 127.0
        scale = torch.clamp(scale, min=1e-12)
        quantized = torch.round(tensor / scale).clamp(-128, 127).to(torch.int8)
        return quantized, scale
    
    @staticmethod
    def dequantize(quantized: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
        return quantized.float() * scale


# Demo: quantization quality analysis
torch.manual_seed(42)
original = torch.randn(256, 4096)  # Simulate a weight or activation tensor

# FP8 quantization
fp8_q, fp8_scale = FP8Quantizer.dynamic_quantize(original)
fp8_deq = FP8Quantizer.dequantize(fp8_q, fp8_scale)
fp8_error = (original - fp8_deq).abs()

# INT8 quantization
int8_q, int8_scale = INT8Quantizer.quantize_per_channel(original)
int8_deq = INT8Quantizer.dequantize(int8_q, int8_scale)
int8_error = (original - int8_deq).abs()

print(f"=== Quantization Quality Analysis ===")
print(f"Original tensor: {original.shape}, range=[{original.min():.2f}, {original.max():.2f}]")
print(f"\n{'Method':<12} {'Max Error':<12} {'Mean Error':<12} {'RMSE':<12} {'Memory (vs FP16)':<18}")
print("-" * 66)

fp16_bytes = original.numel() * 2
fp8_bytes = original.numel() * 1 + 4  # 1 byte per element + scale
int8_bytes = original.numel() * 1 + original.shape[0] * 4  # + per-channel scales

for name, err, mem in [
    ('FP8', fp8_error, fp8_bytes),
    ('INT8', int8_error, int8_bytes)
]:
    print(f"{name:<12} {err.max():.6f}{'':>4} {err.mean():.6f}{'':>4} {err.pow(2).mean().sqrt():.6f}{'':>4} {mem/fp16_bytes:.2f}x")

In [ ]:
# Visualize quantization error distribution

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Original vs quantized values
sample = original[0, :500].numpy()
fp8_sample = fp8_deq[0, :500].numpy()
int8_sample = int8_deq[0, :500].numpy()

axes[0].scatter(sample, fp8_sample, alpha=0.3, s=5, color='blue', label='FP8')
axes[0].scatter(sample, int8_sample, alpha=0.3, s=5, color='red', label='INT8')
axes[0].plot([-3, 3], [-3, 3], 'k--', linewidth=1, label='Perfect')
axes[0].set_xlabel('Original Value')
axes[0].set_ylabel('Quantized Value')
axes[0].set_title('Original vs Quantized')
axes[0].legend()

# Error distribution
axes[1].hist(fp8_error.flatten().numpy(), bins=50, alpha=0.5, color='blue', label='FP8', density=True)
axes[1].hist(int8_error.flatten().numpy(), bins=50, alpha=0.5, color='red', label='INT8', density=True)
axes[1].set_xlabel('Absolute Error')
axes[1].set_ylabel('Density')
axes[1].set_title('Quantization Error Distribution')
axes[1].legend()

# Memory savings
methods = ['FP32', 'FP16', 'FP8', 'INT8', 'INT4']
bytes_per_elem = [4, 2, 1, 1, 0.5]
colors = ['gray', 'steelblue', 'coral', 'green', 'purple']
axes[2].bar(methods, bytes_per_elem, color=colors)
axes[2].set_ylabel('Bytes per Element')
axes[2].set_title('Memory per Element by Precision')

plt.suptitle('Quantization Analysis', fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 5: Communication Kernels for Tensor Parallelism

When models are split across GPUs (tensor parallelism), communication primitives are needed.

In [ ]:
# Simulate tensor parallelism communication patterns

class TPCommunicationSimulator:
    """Simulate tensor parallelism communication patterns.
    
    In real TP, these are NCCL or custom all-reduce operations.
    SGLang uses custom all-reduce kernels that overlap compute+communication.
    """
    
    def __init__(self, world_size: int = 4):
        self.world_size = world_size
    
    def all_reduce_sum(self, shards: List[torch.Tensor]) -> torch.Tensor:
        """All-reduce sum: each GPU gets the sum of all shards.
        
        Used after column-parallel linear layers.
        """
        return sum(shards)
    
    def all_gather(self, shards: List[torch.Tensor], dim: int = -1) -> torch.Tensor:
        """All-gather: concatenate shards along a dimension.
        
        Used for vocabulary-parallel embedding lookup.
        """
        return torch.cat(shards, dim=dim)
    
    def scatter(self, tensor: torch.Tensor, dim: int = -1) -> List[torch.Tensor]:
        """Scatter: split tensor into world_size shards."""
        return torch.chunk(tensor, self.world_size, dim=dim)
    
    def simulate_column_parallel_forward(
        self, x: torch.Tensor, full_weight: torch.Tensor
    ) -> torch.Tensor:
        """Simulate column-parallel linear forward.
        
        Weight is split along output dim. Each GPU computes a shard.
        Results are gathered to form the full output.
        """
        weight_shards = self.scatter(full_weight, dim=0)
        
        # Each GPU computes partial output
        partial_outputs = [x @ shard.T for shard in weight_shards]
        
        # All-gather to get full output
        return self.all_gather(partial_outputs, dim=-1)
    
    def simulate_row_parallel_forward(
        self, x: torch.Tensor, full_weight: torch.Tensor
    ) -> torch.Tensor:
        """Simulate row-parallel linear forward.
        
        Weight is split along input dim. Each GPU has partial input.
        Results are reduced (summed) across GPUs.
        """
        weight_shards = self.scatter(full_weight, dim=1)
        x_shards = self.scatter(x, dim=-1)
        
        # Each GPU computes partial output
        partial_outputs = [x_s @ w_s.T for x_s, w_s in zip(x_shards, weight_shards)]
        
        # All-reduce sum
        return self.all_reduce_sum(partial_outputs)


# Demo: TP communication
tp = TPCommunicationSimulator(world_size=4)

x = torch.randn(16, 4096)         # input
weight = torch.randn(4096, 4096)  # full weight
expected = x @ weight.T           # expected output

# Column parallel
col_result = tp.simulate_column_parallel_forward(x, weight)
col_error = (expected - col_result).abs().max().item()

# Row parallel  
row_result = tp.simulate_row_parallel_forward(x, weight)
row_error = (expected - row_result).abs().max().item()

print(f"=== Tensor Parallelism Communication ===")
print(f"World size: {tp.world_size}")
print(f"Input: {x.shape}, Weight: {weight.shape}")
print(f"Column-parallel error: {col_error:.2e}")
print(f"Row-parallel error:    {row_error:.2e}")

# Communication volume analysis
bytes_per_element = 2  # FP16
seq_len = 2048
hidden = 4096

allreduce_bytes = seq_len * hidden * bytes_per_element * 2  # 2x for ring all-reduce
allgather_bytes = seq_len * hidden * bytes_per_element

print(f"\nCommunication volume per layer (seq={seq_len}, hidden={hidden}, FP16):")
print(f"  All-reduce: {allreduce_bytes/1024/1024:.2f} MB")
print(f"  All-gather: {allgather_bytes/1024/1024:.2f} MB")
print(f"  Per 32 layers: {32 * 2 * allreduce_bytes/1024/1024:.0f} MB")

---
## Part 6: Kernel Dispatch and Selection

SGLang selects the optimal kernel based on hardware, data types, and problem size.

In [ ]:
class KernelDispatcher:
    """Kernel selection and dispatch logic.
    
    SGLang picks the best kernel implementation based on:
    - GPU architecture (sm80, sm89, sm90)
    - Data types (FP16, BF16, FP8)
    - Problem size (batch, seq_len, hidden)
    - Available backends (FlashInfer, FlashAttention, Triton)
    """
    
    KERNEL_REGISTRY = {
        'attention': {
            'flashinfer': {'min_sm': 80, 'dtypes': ['fp16', 'bf16', 'fp8'], 'priority': 1},
            'flash_attn_2': {'min_sm': 80, 'dtypes': ['fp16', 'bf16'], 'priority': 2},
            'triton_attention': {'min_sm': 70, 'dtypes': ['fp16', 'bf16'], 'priority': 3},
            'torch_sdpa': {'min_sm': 0, 'dtypes': ['fp16', 'bf16', 'fp32'], 'priority': 4},
        },
        'gemm': {
            'cublas': {'min_sm': 0, 'dtypes': ['fp16', 'bf16', 'fp32'], 'priority': 2},
            'cutlass_fp8': {'min_sm': 89, 'dtypes': ['fp8'], 'priority': 1},
            'marlin_int4': {'min_sm': 80, 'dtypes': ['int4'], 'priority': 1},
        },
        'activation': {
            'triton_silu_mul': {'min_sm': 70, 'dtypes': ['fp16', 'bf16'], 'priority': 1},
            'cuda_silu_mul': {'min_sm': 70, 'dtypes': ['fp16', 'bf16'], 'priority': 2},
            'torch_silu_mul': {'min_sm': 0, 'dtypes': ['fp16', 'bf16', 'fp32'], 'priority': 3},
        },
        'rmsnorm': {
            'triton_rmsnorm': {'min_sm': 70, 'dtypes': ['fp16', 'bf16'], 'priority': 1},
            'cuda_rmsnorm': {'min_sm': 70, 'dtypes': ['fp16', 'bf16'], 'priority': 2},
            'torch_rmsnorm': {'min_sm': 0, 'dtypes': ['fp16', 'bf16', 'fp32'], 'priority': 3},
        },
    }
    
    def __init__(self, gpu_sm: int = 80, available_backends: List[str] = None):
        self.gpu_sm = gpu_sm
        self.available_backends = set(available_backends or [
            'flashinfer', 'cublas', 'triton_silu_mul', 'triton_rmsnorm',
            'torch_sdpa', 'torch_silu_mul', 'torch_rmsnorm'
        ])
    
    def select_kernel(self, op_type: str, dtype: str) -> Optional[str]:
        """Select the best available kernel for an operation."""
        if op_type not in self.KERNEL_REGISTRY:
            return None
        
        candidates = []
        for kernel_name, spec in self.KERNEL_REGISTRY[op_type].items():
            if (kernel_name in self.available_backends and
                self.gpu_sm >= spec['min_sm'] and
                dtype in spec['dtypes']):
                candidates.append((kernel_name, spec['priority']))
        
        if not candidates:
            return None
        
        # Return highest priority (lowest number)
        candidates.sort(key=lambda x: x[1])
        return candidates[0][0]
    
    def dispatch_table(self, dtype: str = 'fp16') -> dict:
        """Generate full dispatch table."""
        table = {}
        for op_type in self.KERNEL_REGISTRY:
            kernel = self.select_kernel(op_type, dtype)
            table[op_type] = kernel or 'NOT AVAILABLE'
        return table


# Demo: kernel dispatch for different GPU architectures
gpu_configs = [
    ('A100 (sm80)', 80),
    ('L40S (sm89)', 89),
    ('H100 (sm90)', 90),
    ('T4 (sm75)', 75),
]

for gpu_name, sm in gpu_configs:
    dispatcher = KernelDispatcher(gpu_sm=sm)
    table = dispatcher.dispatch_table('fp16')
    print(f"\n{gpu_name}:")
    for op, kernel in table.items():
        print(f"  {op:.<20s} {kernel}")

# FP8 dispatch
print(f"\n=== FP8 Dispatch (H100) ===")
h100 = KernelDispatcher(
    gpu_sm=90, 
    available_backends=['flashinfer', 'cutlass_fp8', 'cublas', 'triton_silu_mul', 'torch_rmsnorm']
)
for op, kernel in h100.dispatch_table('fp8').items():
    print(f"  {op:.<20s} {kernel}")

In [ ]:
# Profile: kernel performance across different batch sizes

def benchmark_across_sizes(
    fn, name: str, sizes: List[Tuple[int, int]], n_iters: int = 100
) -> List[float]:
    """Benchmark a function across different tensor sizes."""
    times = []
    for seq_len, hidden in sizes:
        x = torch.randn(seq_len, hidden, device=DEVICE)
        # Warmup
        for _ in range(5):
            fn(x)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        
        start = time.time()
        for _ in range(n_iters):
            fn(x)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        times.append((time.time() - start) / n_iters * 1000)
    
    return times


# RMSNorm benchmark
weight = torch.ones(4096, device=DEVICE)

def rmsnorm_op(x):
    return rmsnorm_unfused(x, weight[:x.shape[-1]])

sizes = [(64, 4096), (256, 4096), (1024, 4096), (4096, 4096)]
times = benchmark_across_sizes(rmsnorm_op, 'RMSNorm', sizes)

print(f"=== RMSNorm Benchmark ===")
print(f"{'Size':<20} {'Time (ms)':<12} {'Throughput (GB/s)':<20}")
print("-" * 52)
for (seq, hid), t in zip(sizes, times):
    bytes_processed = seq * hid * 4 * 2  # read + write, float32
    throughput = bytes_processed / (t / 1000) / 1e9
    print(f"({seq:>4}, {hid:>4}){'':>8} {t:<12.4f} {throughput:<20.1f}")

---
## Part 7: Triton Kernel — Fused RMSNorm + Residual

In [ ]:
# Triton kernel for fused RMSNorm + residual add

if HAS_TRITON:
    @triton.jit
    def rmsnorm_residual_kernel(
        x_ptr, residual_ptr, weight_ptr, output_ptr, hidden_out_ptr,
        hidden_size,
        eps: tl.constexpr,
        BLOCK_SIZE: tl.constexpr,
    ):
        """Fused residual add + RMSNorm.
        
        Each program processes one row (one token).
        """
        row_idx = tl.program_id(0)
        row_start = row_idx * hidden_size
        
        # Load and compute hidden = x + residual
        variance = tl.zeros([1], dtype=tl.float32)
        for off in range(0, hidden_size, BLOCK_SIZE):
            cols = off + tl.arange(0, BLOCK_SIZE)
            mask = cols < hidden_size
            x = tl.load(x_ptr + row_start + cols, mask=mask).to(tl.float32)
            res = tl.load(residual_ptr + row_start + cols, mask=mask).to(tl.float32)
            hidden = x + res
            # Store hidden for residual stream
            tl.store(hidden_out_ptr + row_start + cols, hidden, mask=mask)
            variance += tl.sum(hidden * hidden)
        
        variance = variance / hidden_size
        rstd = 1.0 / tl.sqrt(variance + eps)
        
        # Normalize and scale
        for off in range(0, hidden_size, BLOCK_SIZE):
            cols = off + tl.arange(0, BLOCK_SIZE)
            mask = cols < hidden_size
            hidden = tl.load(hidden_out_ptr + row_start + cols, mask=mask).to(tl.float32)
            w = tl.load(weight_ptr + cols, mask=mask).to(tl.float32)
            output = hidden * rstd * w
            tl.store(output_ptr + row_start + cols, output, mask=mask)
    
    print("Triton fused RMSNorm + residual kernel defined.")
    print("This kernel reads x and residual once, computes add + norm, writes output and hidden.")
else:
    print("Triton not available. The fused kernel would:")
    print("1. Read x[i] and residual[i] from global memory (once)")
    print("2. Compute hidden[i] = x[i] + residual[i] in registers")
    print("3. Compute variance using shared memory reduction")
    print("4. Normalize: output[i] = hidden[i] * rsqrt(var + eps) * weight[i]")
    print("5. Write output[i] and hidden[i] to global memory (once)")
    print("\nMemory accesses: 2 reads + 2 writes (vs 5+ reads + 4+ writes unfused)")

In [ ]:
# Comprehensive kernel performance comparison

hidden_sizes = [1024, 2048, 4096, 8192]
batch_sizes = [1, 8, 32, 128, 512]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Scaling with batch size (fixed hidden=4096)
fused_times = []
unfused_times = []
weight = torch.ones(4096, device=DEVICE)

for bs in batch_sizes:
    x = torch.randn(bs, 4096, device=DEVICE)
    res = torch.randn(bs, 4096, device=DEVICE)
    
    # Warmup
    for _ in range(10):
        _ = rmsnorm_with_residual_unfused(x, res, weight)
        _ = rmsnorm_with_residual_fused(x, res, weight)
    
    n_iters = 200
    start = time.time()
    for _ in range(n_iters):
        _ = rmsnorm_with_residual_unfused(x, res, weight)
    unfused_times.append((time.time() - start) / n_iters * 1000)
    
    start = time.time()
    for _ in range(n_iters):
        _ = rmsnorm_with_residual_fused(x, res, weight)
    fused_times.append((time.time() - start) / n_iters * 1000)

axes[0].plot(batch_sizes, unfused_times, 'r-o', label='Unfused (5 ops)', linewidth=2)
axes[0].plot(batch_sizes, fused_times, 'b-o', label='Fused (1 op)', linewidth=2)
axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Time (ms)')
axes[0].set_title('RMSNorm+Residual: Scaling with Batch Size\n(hidden=4096)')
axes[0].legend()
axes[0].set_xscale('log')

# 2. Memory bandwidth analysis
bytes_read_unfused = [bs * 4096 * 4 * 5 for bs in batch_sizes]  # 5 reads
bytes_read_fused = [bs * 4096 * 4 * 2 for bs in batch_sizes]    # 2 reads

bw_unfused = [br / (t/1000) / 1e9 for br, t in zip(bytes_read_unfused, unfused_times)]
bw_fused = [br / (t/1000) / 1e9 for br, t in zip(bytes_read_fused, fused_times)]

axes[1].bar([i-0.2 for i in range(len(batch_sizes))], bw_unfused, 0.35, 
            label='Unfused', color='salmon')
axes[1].bar([i+0.2 for i in range(len(batch_sizes))], bw_fused, 0.35,
            label='Fused', color='steelblue')
axes[1].set_xticks(range(len(batch_sizes)))
axes[1].set_xticklabels(batch_sizes)
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Effective Bandwidth (GB/s)')
axes[1].set_title('Memory Bandwidth Utilization')
axes[1].legend()

plt.suptitle('Custom Kernel Performance Analysis', fontsize=14)
plt.tight_layout()
plt.show()

---
## Hands-On Assignments

### Assignment 1: Write a Fused RMSNorm + Residual Kernel

**Task**: Implement the fused RMSNorm + residual add operation.

Requirements:
1. Implement in either Triton (if available) or pure PyTorch with fusion hints
2. Handle arbitrary hidden sizes (not just powers of 2)
3. Support both FP16 and FP32
4. Verify correctness against the unfused version
5. Benchmark across batch sizes and hidden sizes

In [ ]:
# === ASSIGNMENT 1: Fused RMSNorm + Residual ===

def fused_rmsnorm_residual(
    x: torch.Tensor, residual: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    TODO: Implement fused residual add + RMSNorm.
    
    Input: x (seq, hidden), residual (seq, hidden), weight (hidden,)
    Output: (normalized, hidden_state) where hidden_state = x + residual
    
    Requirements:
    - Single-pass computation
    - Support FP16 and FP32
    - Handle arbitrary hidden sizes
    """
    # YOUR CODE HERE
    pass

print("Assignment 1: Implement fused_rmsnorm_residual().")
print("Verify correctness and benchmark vs unfused version.")

### Assignment 2: Profile and Compare Kernel Implementations

**Task**: Build a comprehensive kernel profiling framework.

Requirements:
1. Profile at least 3 operations: RMSNorm, SiLU×mul, attention
2. Measure: latency, throughput (TFLOPS), bandwidth (GB/s)
3. Vary: batch size, hidden size, sequence length
4. Compare fused vs unfused for each operation
5. Generate a report with roofline model analysis

In [ ]:
# === ASSIGNMENT 2: Kernel Profiling Framework ===

class KernelProfiler:
    """
    TODO: Build a comprehensive kernel profiler.
    
    Measure: latency, TFLOPS, bandwidth.
    Compare: fused vs unfused implementations.
    Output: tables and roofline charts.
    """
    def __init__(self, device: str = 'cpu'):
        # YOUR CODE HERE
        pass
    
    def profile_op(self, fn, input_shapes, flops_fn, bytes_fn, n_iters=100):
        # YOUR CODE HERE
        pass
    
    def report(self) -> str:
        # YOUR CODE HERE
        pass

print("Assignment 2: Build KernelProfiler with roofline analysis.")

### Assignment 3: Add a New Custom Kernel to the Dispatch Table

**Task**: Extend the kernel dispatch system with a new fused operation.

Requirements:
1. Implement a fused `LayerNorm + Dropout + Residual` kernel
2. Register it in the KernelDispatcher with proper metadata
3. Implement fallback logic (use unfused if fused unavailable)
4. Test dispatch across different GPU architectures
5. Benchmark against the unfused sequence

In [ ]:
# === ASSIGNMENT 3: Custom Kernel Dispatch ===

class ExtendedKernelDispatcher(KernelDispatcher):
    """
    TODO: Extend KernelDispatcher with a new fused operation.
    
    Add: fused LayerNorm + Dropout + Residual
    Register in KERNEL_REGISTRY
    Implement with fallback
    """
    def __init__(self, gpu_sm: int = 80, available_backends: List[str] = None):
        super().__init__(gpu_sm, available_backends)
        # YOUR CODE: register new kernel
    
    def fused_ln_dropout_residual(self, x, residual, weight, bias, p=0.1):
        # YOUR CODE HERE
        pass

print("Assignment 3: Extend the dispatch table with a new fused kernel.")

---
## Summary

| Kernel Category | Examples | Speedup Source |
|----------------|----------|----------------|
| Attention | FlashInfer, FlashAttention | O(N) memory, tiling |
| Fused Activations | SiLU×mul, GeGLU | Reduced memory bandwidth |
| Fused Norms | RMSNorm + residual | Single read/write pass |
| Quantization | FP8/INT8 GEMM | Lower precision = more TFLOPS |
| Communication | Custom all-reduce | Overlap compute + comms |

### Key Takeaways
1. Custom kernels are **memory-bandwidth-bound** — fusion reduces reads/writes
2. FlashInfer is SGLang's primary attention backend (supports paged KV cache)
3. Triton makes it easy to write custom fused kernels without raw CUDA
4. Kernel dispatch selects the best implementation for the hardware
5. FP8 quantization on H100 gives near 2x speedup with minimal quality loss

In [ ]:
print("=== End of Chapter 6.8: Custom CUDA Kernels in SGLang ===")
print("\nKey kernel locations in SGLang:")
print("  Attention:    sglang/srt/layers/attention/")
print("  Activations:  sglang/srt/layers/activation.py")
print("  Norms:        sglang/srt/layers/layernorm.py")
print("  Quantization: sglang/srt/layers/quantization/")
print("  Custom ops:   sglang/srt/custom_op.py")